# Soft Computing Project — Jigsaw Toxic Comment Feature Selection

**Purpose:** A portable Kaggle/Google Colab notebook for demonstrating and reproducing the completed project.

### Project pipeline
Raw comments → text cleaning → TF-IDF → **10,000 numerical features** → evolutionary feature selection → **exactly 500 features** → LinearSVC → F1 / FPR / inference time.

### Final algorithms
NSGA-II · SPEA2 · MOEA/D · MODE

### Final objectives
1. Maximize F1 score
2. Minimize False Positive Rate (FPR)
3. Minimize inference time

> **Demo recommendation:** use the existing sparse `.npz` matrices and saved 500-feature index files. Do not upload the 2.6 GB dense CSV to Colab/Kaggle just to run the model. The notebook can generate a small human-readable CSV preview from the sparse matrix.


## 1. Environment setup

Run this cell once in Kaggle or Colab.


In [ ]:
import sys, subprocess, importlib.util
packages = ["numpy", "scipy", "pandas", "scikit-learn", "matplotlib", "pymoo"]
for pkg, mod in [("numpy","numpy"),("scipy","scipy"),("pandas","pandas"),("scikit-learn","sklearn"),("matplotlib","matplotlib"),("pymoo","pymoo")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


## 2. Data location

### What to upload/add to the notebook environment

The preferred runtime files are the efficient sparse artifacts:

- `X_train.npz`
- `X_val.npz`
- `y_train.npy`
- `y_val.npy`
- `feature_names.txt`

For the fast final-model demonstration also add:

- `selected_features_nsga2_balanced.npy`
- `selected_features_spea2_balanced.npy`
- `selected_features_moead_balanced.npy`
- `selected_features_mode_balanced.npy` (when available)
- optionally `final_pareto_solutions.csv`
- optionally `final_algorithm_comparison.csv`

**Kaggle:** add the project-data directory as a Kaggle Dataset and point `DATA_DIR` to it.

**Colab:** upload the files or mount Google Drive and point `DATA_DIR` to the folder.


In [ ]:
from pathlib import Path
import os

candidates = [
    Path("/kaggle/input/sc-project-data"),
    Path("/kaggle/input"),
    Path("/content/sc_project_data"),
    Path("/content"),
    Path(".")
]

DATA_DIR = None
required = ["X_train.npz","X_val.npz","y_train.npy","y_val.npy","feature_names.txt"]
for root in candidates:
    if all((root / f).exists() for f in required):
        DATA_DIR = root
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find the project artifacts. Set DATA_DIR manually to the folder containing "
        + ", ".join(required)
    )

print("Using:", DATA_DIR)


## 3. Load the numerical dataset

The `.npz` files keep the TF-IDF matrices sparse, so they are much more suitable for Kaggle/Colab than the multi-GB dense CSV exports.


In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

X_train = sparse.load_npz(DATA_DIR / "X_train.npz")
X_val = sparse.load_npz(DATA_DIR / "X_val.npz")
y_train = np.load(DATA_DIR / "y_train.npy")
y_val = np.load(DATA_DIR / "y_val.npy")

feature_names = [line.strip() for line in open(DATA_DIR / "feature_names.txt", encoding="utf-8") if line.strip()]

print("X_train:", X_train.shape, X_train.dtype)
print("X_val  :", X_val.shape, X_val.dtype)
print("y_train:", y_train.shape, np.unique(y_train, return_counts=True))
print("y_val  :", y_val.shape, np.unique(y_val, return_counts=True))
print("Features:", len(feature_names))
print("Train sparsity:", 1 - X_train.nnz / X_train.size)
print("Val sparsity  :", 1 - X_val.nnz / X_val.size)


## 4. Generate a small processed CSV for inspection

This creates a **100-row, 10,001-column** human-readable sample:

- 10,000 TF-IDF numeric columns
- 1 `target` column
- every TF-IDF column is prefixed with `tfidf_`

This is for inspection/demo purposes. The full CSVs remain large because the sparse matrix is being written in dense tabular form.


In [ ]:
sample_path = DATA_DIR / "processed_sample.csv"
sample_rows = 100
feature_cols = ["tfidf_" + f for f in feature_names]

sample = X_train[:sample_rows].toarray()
sample_df = pd.DataFrame(sample, columns=feature_cols)
sample_df["target"] = y_train[:sample_rows]
sample_df.to_csv(sample_path, index=False)

print("Saved:", sample_path)
print("Shape :", sample_df.shape)
display(sample_df.iloc[:5, :8])


## 5. Baseline model

The reference model uses all 10,000 TF-IDF features:

`LinearSVC(class_weight="balanced", C=1.0, random_state=42)`


In [ ]:
import time
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score, precision_score, recall_score

baseline = LinearSVC(class_weight="balanced", C=1.0, random_state=42, dual="auto")
t0 = time.perf_counter()
baseline.fit(X_train, y_train)
train_time = time.perf_counter() - t0

times = []
for _ in range(5):
    t0 = time.perf_counter()
    pred = baseline.predict(X_val)
    times.append(time.perf_counter() - t0)

cm = confusion_matrix(y_val, pred)
tn, fp, fn, tp = cm.ravel()
f1 = f1_score(y_val, pred)
fpr = fp / (fp + tn)

baseline_result = {
    "Model": "Baseline",
    "Features": 10000,
    "F1": f1,
    "FPR": fpr,
    "Inference_ms": np.mean(times) * 1000,
    "Accuracy": accuracy_score(y_val, pred),
    "Precision": precision_score(y_val, pred),
    "Recall": recall_score(y_val, pred),
}
pd.DataFrame([baseline_result])


### Reference baseline from the completed experiment

The completed local experiment reported:

- **F1 = 0.6848**
- **FPR ≈ 6.81%**
- **Accuracy = 92.19%**
- **Precision = 0.5809**
- **Recall = 0.8339**
- **Inference = 3.375 ms** for 31,915 validation comments

Inference timing can differ on Kaggle/Colab because hardware and runtime conditions differ. The classification metrics should remain the primary reference.


## 6. Evaluate the saved 500-feature solutions

This is the quickest way to demonstrate the actual final models. Each saved feature set was produced by the corresponding evolutionary algorithm and contains exactly 500 feature indices.


In [ ]:
from pathlib import Path

balanced_files = {
    "NSGA-II": DATA_DIR / "selected_features_nsga2_balanced.npy",
    "SPEA2": DATA_DIR / "selected_features_spea2_balanced.npy",
    "MOEA/D": DATA_DIR / "selected_features_moead_balanced.npy",
    "MODE": DATA_DIR / "selected_features_mode_balanced.npy",
}

reference_results = {
    "NSGA-II": {"F1": 0.5911, "FPR": 0.0743, "Inference_ms": 1.423},
    "SPEA2":  {"F1": 0.5727, "FPR": 0.0658, "Inference_ms": 1.267},
    "MOEA/D": {"F1": 0.6395, "FPR": 0.0731, "Inference_ms": 2.671},
    "MODE":   {"F1": 0.5520, "FPR": 0.0791, "Inference_ms": 1.558},
}

rows = []
for algo, p in balanced_files.items():
    if not p.exists():
        print(f"{algo}: balanced feature file not found -> add {p.name} to the runtime files")
        continue
    idx = np.load(p).astype(int)
    assert len(idx) == 500 and len(np.unique(idx)) == 500
    Xtr = X_train[:, idx]
    Xv = X_val[:, idx]
    clf = LinearSVC(class_weight="balanced", C=1.0, random_state=42, dual="auto")
    clf.fit(Xtr, y_train)
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        pred = clf.predict(Xv)
        times.append(time.perf_counter() - t0)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    rows.append({
        "Algorithm": algo,
        "Features": len(idx),
        "F1": f1_score(y_val, pred),
        "FPR": fp / (fp + tn),
        "Inference_ms": np.mean(times) * 1000,
        "Precision": precision_score(y_val, pred),
        "Recall": recall_score(y_val, pred),
    })

final_models = pd.DataFrame(rows)
final_models


## 7. Completed-experiment reference comparison

These are the **final corrected results** from the local evolutionary experiment. They are included as a reference so the notebook output can be checked against the completed run.

| Method | Features | Balanced F1 | Balanced FPR | Balanced inference |
|---|---:|---:|---:|---:|
| Baseline | 10,000 | 0.6848 | 6.81% | 3.375 ms* |
| NSGA-II | 500 | **0.5911** | 7.43% | 1.423 ms* |
| SPEA2 | 500 | 0.5727 | **6.58%** | **1.267 ms*** |
| MOEA/D | 500 | **0.6395** | 7.31% | 2.671 ms* |
| MODE | 500 | 0.5520 | 7.91% | 1.558 ms* |

`*` Timing depends on machine/runtime; the local completed experiment used the values shown above.

### Final conclusion from the corrected analysis

**NSGA-II was selected as the overall algorithm** because its balanced solution had the smallest normalized Euclidean distance to the ideal point across all three objectives and it also achieved the highest hypervolume.

MOEA/D produced the **highest individual F1**, while SPEA2 produced the **lowest FPR**. This is exactly why the problem is multi-objective: there is no single metric that every algorithm wins.


## 8. Visual comparison


In [ ]:
import matplotlib.pyplot as plt

if len(final_models):
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    ax[0].bar(final_models["Algorithm"], final_models["F1"])
    ax[0].set_title("F1 Score")
    ax[0].set_ylim(0, 0.75)
    ax[0].tick_params(axis="x", rotation=30)

    ax[1].bar(final_models["Algorithm"], final_models["FPR"] * 100)
    ax[1].set_title("FPR (%)")
    ax[1].tick_params(axis="x", rotation=30)

    ax[2].bar(final_models["Algorithm"], final_models["Inference_ms"])
    ax[2].set_title("Inference Time (ms)")
    ax[2].tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()


## 9. Optional: inspect the actual Pareto solutions

If `final_pareto_solutions.csv` is available, load it here to inspect the final non-dominated solutions discovered by the four algorithms.


In [ ]:
pareto_path = DATA_DIR / "final_pareto_solutions.csv"
if pareto_path.exists():
    pareto = pd.read_csv(pareto_path)
    print("Shape:", pareto.shape)
    display(pareto.head())
    print("Algorithms:", pareto["Algorithm"].value_counts().to_dict() if "Algorithm" in pareto.columns else "Check column names.")
else:
    print("final_pareto_solutions.csv not found. Add it to inspect the full audited Pareto front.")


## 10. Optional: run the full evolutionary search

The completed local search used:

- population = 28
- generations = 20
- exact feature count = 500
- fixed stratified search subset = 8,000 validation samples
- 3 objectives: `1-F1`, FPR, inference time
- same LinearSVC evaluator for all four methods

The completed search took about **16.3 minutes locally**. For a live demonstration, the saved 500-feature solutions above are preferable because they let you demonstrate the final models quickly.

If you have copied the original `run_evolutionary_experiment.py` into the runtime environment, you can execute it from a shell cell instead of rebuilding the experiment here.


In [ ]:
# Optional full rerun — this can take ~16+ minutes depending on hardware.
# Only run this cell when a full evolutionary search is actually required.

runner = DATA_DIR / "run_evolutionary_experiment.py"
if runner.exists():
    print("Runner found:", runner)
    # Uncomment in Kaggle/Colab when you intentionally want the full search:
    # !python "{runner}"
else:
    print("Original evolutionary runner not found. Use the saved final artifacts for the fast demonstration.")


## 11. What to say during the demonstration

1. **Why TF-IDF?**  
   "Our original dataset is text. TF-IDF converts each comment into a numerical feature vector that the classifier and evolutionary algorithms can operate on."

2. **Why 10,000 features?**  
   "It gives us a high-dimensional numerical representation from which the evolutionary algorithms can perform meaningful feature selection."

3. **Why exactly 500?**  
   "The professor required a reduction to around 500 features. We enforce exactly 500 so all four algorithms are compared at the same feature budget."

4. **Why three objectives?**  
   "We want good toxic detection, fewer false alarms, and fast prediction, so we optimize F1, FPR and inference time simultaneously."

5. **Why NSGA-II?**  
   "NSGA-II gave the best overall balanced multi-objective solution and the highest hypervolume, while MOEA/D and SPEA2 each excelled at individual objectives."

6. **Why not use the 2.6 GB CSV?**  
   "The CSV is a human-readable export for inspection. The actual ML pipeline uses the much more efficient sparse `.npz` representation."
